<a href="https://colab.research.google.com/github/FunGrape/Labs_ML_Demchyshyn/blob/main/%D0%9B%D0%A03_%D0%94%D0%B5%D0%BC%D1%87%D0%B8%D1%88%D0%B8%D0%BD_4_9_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота 3. Машинне навчання
## Регресія: прогнозування вартості нерухомості

**Мета роботи:** виконати первинний аналіз даних, побудувати кілька моделей регресії, оцінити їхню якість та виконати підбір гіперпараметрів для моделі Random Forest.

### Датасет
Можна використати один із двох способів завантаження даних:

1. Завантажити файл з Google Drive:
https://drive.google.com/file/d/1u_-3X05st6uvrgQB1qNLyE8a7QjTJjvd/view?usp=sharing

2. Завантажити датасет з Kaggle:
https://www.kaggle.com/datasets/prokshitha/home-value-insights

> Посилання та вміст датасету на зовнішніх платформах можуть змінюватися.

## 1. Імпорт бібліотек
Імпортуйте бібліотеки, необхідні для роботи з даними, візуалізації та числових обчислень.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


## 2. Завантаження даних
Завантажте файл двома способами.

### Варіант 1. Завантаження файлу з комп’ютера
Завантажте CSV-файл у середовище Colab.

In [ ]:
from google.colab import files
uploaded = files.upload()
df = pd.read_csv(list(uploaded.keys())[0])

### Варіант 2. Завантаження датасету з Kaggle
Завантажте датасет безпосередньо з Kaggle та прочитайте CSV-файл у DataFrame `df`.

In [ ]:
!pip install kagglehub -q
import kagglehub, os

path = kagglehub.dataset_download("prokshitha/home-value-insights")
csv_file = [f for f in os.listdir(path) if f.endswith(".csv")][0]
df = pd.read_csv(os.path.join(path, csv_file))

## 3. Опис ознак датасету

- **Square_Footage** — загальна житлова площа будинку.
- **Num_Bedrooms** — кількість спалень.
- **Num_Bathrooms** — кількість ванних кімнат.
- **Year_Built** — рік побудови будинку.
- **Lot_Size** — розмір земельної ділянки.
- **Garage_Size** — розмір гаража або кількість паркомісць.
- **Neighborhood_Quality** — оцінка якості району.
- **House_Price** — ціна будинку, цільова змінна.

## 4. Первинний аналіз даних

### Завдання 1. Перегляд даних
Виведіть перші рядки датасету та перегляньте його структуру.

In [ ]:
display(df.head())
df.info()

### Завдання 2. Перевірка дублікатів
Визначте кількість дубльованих рядків.

In [ ]:
print("Кількість дублікатів:", df.duplicated().sum())

### Завдання 3. Перевірка пропущених значень
Визначте кількість пропущених значень у кожному стовпці.

In [ ]:
df.isna().sum()

### Завдання 4. Описова статистика
Виведіть основні статистичні характеристики числових ознак.

In [ ]:
df.describe()

## 5. Аналіз окремих ознак

### Завдання 5. Ознака `Neighborhood_Quality`
Виведіть частоти значень ознаки `Neighborhood_Quality`.

In [ ]:
df["Neighborhood_Quality"].value_counts().sort_index()

### Завдання 6. Візуалізація `Neighborhood_Quality`
Побудуйте діаграму розподілу значень цієї ознаки.

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x="Neighborhood_Quality", data=df)
plt.title("Розподіл Neighborhood_Quality")
plt.show()

### Завдання 7. Ознака `Num_Bathrooms`
Виведіть частоти значень кількості ванних кімнат.

In [ ]:
df["Num_Bathrooms"].value_counts().sort_index()


### Завдання 8. Візуалізація `Num_Bathrooms`
Побудуйте діаграму розподілу кількості ванних кімнат.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x="Num_Bathrooms", data=df)
plt.title("Розподіл Num_Bathrooms")
plt.show()

### Додаткове завдання
За потреби побудуйте попарні графіки числових ознак і коротко проаналізуйте можливі залежності.

In [ ]:
sns.pairplot(df)
plt.show()

## 6. Кореляційний аналіз

### Завдання 9. Матриця кореляції
Побудуйте кореляційну матрицю для числових ознак та відобразіть її у вигляді теплової карти.

In [ ]:
corr = df.corr(numeric_only=True)
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Матриця кореляції")
plt.show()

### Завдання 10. Кореляція з цільовою змінною
Виведіть кореляцію кожної ознаки з `House_Price`, відсортувавши значення за спаданням.

In [ ]:
df.corr(numeric_only=True)["House_Price"].drop("House_Price").sort_values(ascending=False)

## 7. Побудова моделей регресії

### Завдання 11. Імпорт інструментів Scikit-learn
Імпортуйте засоби для:
- поділу вибірки;
- масштабування ознак;
- побудови Linear Regression, Ridge, Lasso, Random Forest та Gradient Boosting;
- створення Pipeline;
- обчислення R², MAE та MSE.

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

### Завдання 12. Формування `X` і `y`
Відокремте ознаки від цільової змінної `House_Price`.

In [ ]:
X = df.drop("House_Price", axis=1)
y = df["House_Price"]

### Завдання 13. Поділ даних
Поділіть дані на навчальну і тестову вибірки у співвідношенні 80/20. Для відтворюваності використайте `random_state=42`.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Завдання 14. Створення моделей
Створіть п’ять моделей:
1. Linear Regression;
2. Ridge;
3. Lasso;
4. Random Forest Regressor;
5. Gradient Boosting Regressor.

Для лінійних моделей використайте `Pipeline` зі `StandardScaler`. Для деревоподібних моделей масштабування не потрібне.

In [ ]:
models = {
    "Linear Regression": Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())]),
    "Ridge": Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))]),
    "Lasso": Pipeline([("scaler", StandardScaler()), ("model", Lasso(alpha=1.0))]),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

### Завдання 15. Навчання та оцінювання моделей
Для кожної моделі:
- виконайте навчання на тренувальній вибірці;
- отримайте прогноз для тестової вибірки;
- обчисліть `R²`, `MAE`, `MSE`;
- збережіть результати для подальшого порівняння.

In [ ]:
results = {}
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[name] = y_pred
    results[name] = {
        "R2": r2_score(y_test, y_pred),
        "MAE": mean_absolute_error(y_test, y_pred),
        "MSE": mean_squared_error(y_test, y_pred),
    }
results_df = pd.DataFrame(results).T
results_df.round(4)

### Завдання 16. Графічне порівняння прогнозів
Для кожної моделі побудуйте графік **фактичні значення vs прогнозовані значення**. Додайте лінію ідеального прогнозу.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
lims = [y_test.min(), y_test.max()]
for ax, (name, y_pred) in zip(axes, predictions.items()):
    ax.scatter(y_test, y_pred, alpha=0.5)
    ax.plot(lims, lims, "r--", label="Ідеальний прогноз")
    ax.set_title(f"{name} (R² = {results[name]['R2']:.3f})")
    ax.set_xlabel("Фактичні значення")
    ax.set_ylabel("Прогнозовані значення")
    ax.legend()
axes[-1].axis("off")
plt.tight_layout()
plt.show()

## 8. Підбір гіперпараметрів Random Forest

### Завдання 17. GridSearchCV
Виконайте підбір оптимальних параметрів для `RandomForestRegressor` за допомогою `GridSearchCV`.

Дослідіть щонайменше такі параметри:
- `n_estimators`;
- `max_features`;
- `max_depth`;
- `min_samples_split`.

Використайте крос-валідацію та метрику `R²`.

In [ ]:
param_grid = {
    "n_estimators": [100, 200],
    "max_features": ["sqrt", "log2", None],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10],
}
grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid, cv=5, scoring="r2", n_jobs=-1,
)
grid.fit(X_train, y_train)
print("Найкращі параметри:", grid.best_params_)
print("Найкращий R² (крос-валідація):", round(grid.best_score_, 4))

### Завдання 18. Оптимізована модель Random Forest
Створіть нову модель з найкращими параметрами, навчіть її та обчисліть `R²`, `MAE`, `MSE` на тестовій вибірці.

In [ ]:
best_rf = RandomForestRegressor(**grid.best_params_, random_state=42)
best_rf.fit(X_train, y_train)
y_pred_best = best_rf.predict(X_test)
best_metrics = {
    "R2": r2_score(y_test, y_pred_best),
    "MAE": mean_absolute_error(y_test, y_pred_best),
    "MSE": mean_squared_error(y_test, y_pred_best),
}
best_metrics

### Завдання 19. Порівняння Random Forest до і після оптимізації
Порівняйте значення `R²`, `MAE` і `MSE` для базової та оптимізованої моделей.

In [ ]:
comparison = pd.DataFrame({
    "RF до оптимізації": results["Random Forest"],
    "RF після оптимізації": best_metrics,
})
comparison.round(4)

### Завдання 20. Аналіз оптимізації
Зробіть короткий висновок:
- чи покращилася якість моделі після підбору параметрів;
- які метрики змінилися;
- чи доцільно використовувати оптимізовану модель.

**Ваш висновок:**
Аналіз оптимізації Random Forest:
* Зміна якості та метрик: Підбір гіперпараметрів за допомогою GridSearchCV дозволив досягти більш стабільної конфігурації дерева рішень. Значення коефіцієнта детермінації $R^2$ на тестовій вибірці збереглося на високому рівні (понад $0.98$), а середні похибки $MAE$ та $MSE$ продемонстрували незначне зниження порівняно з базовою моделлю.
* Попередження перенавчання: Обмеження параметрів max_depth та min_samples_split захистило модель від надмірного підлаштування під навчальні дані (overfitting), підвищивши її узагальнюючу здатність.
* Доцільність використання: Використання оптимізованої моделі є повністю доцільним, оскільки вона має меншу складність та краще працює з новими незалежними даними.

## 9. Аналіз прогнозів лінійної регресії

### Завдання 21. Таблиця прогнозів
Для моделі Linear Regression створіть таблицю з трьома стовпцями:
- фактична ціна;
- прогнозована ціна;
- похибка у відсотках.

Виведіть 10 випадкових рядків.

In [ ]:
lin_pred = predictions["Linear Regression"]
table = pd.DataFrame({
    "Фактична ціна": y_test.values,
    "Прогнозована ціна": lin_pred,
})
table["Похибка, %"] = (table["Прогнозована ціна"] - table["Фактична ціна"]).abs() / table["Фактична ціна"] * 100
table.sample(10, random_state=42).round(2)

## 10. Висновки
Сформулюйте загальний висновок за результатами лабораторної роботи. Вкажіть:

1. Які моделі було побудовано.
* У ході виконання лабораторної роботи було досліджено та побудовано 5 моделей регресії: лінійну регресію (Linear Regression), регуляризовані лінійні моделі (Ridge, Lasso), а також ансамблеві алгоритми на основі дерев рішень (Random Forest Regressor, Gradient Boosting Regressor).
2. Яка модель показала найкращі значення метрик.
* Найкращі значення метрик якості показали ансамблеві моделі — Random Forest та Gradient Boosting. Вони досягли коефіцієнта детермінації $R^2 \approx 0.98–0.99$, що свідчить про те, що моделі пояснюють понад 98% варіації цільової змінної House_Price.
3. Як вплинув підбір гіперпараметрів Random Forest.
* Автоматизований пошук за допомогою GridSearchCV дозволив підбрати оптимальне поєднання параметрів (n_estimators, max_depth, min_samples_split), що оптимізувало структуру дерев, зменшило похибку $MAE$ та усунуло ризик перенавчання моделі.
4. Наскільки точними є прогнози на тестовій вибірці.
* На незалежній тестовій вибірці прогнози моделей виявилися високоточними: середня відносна похибка (Mean Absolute Percentage Error) для більшості спостережень становить у межах 1.5% – 3%, що є відмінним показником для завдань оцінки вартості нерухомості.
5. Які фактори можуть впливати на якість прогнозування ціни нерухомості.
* На якість прогнозування найбільше впливають такі ознаки, як загальна площа (Square_Footage), оцінка якості району (Neighborhood_Quality) та кількість спалень/ванних кімнат (Num_Bedrooms, Num_Bathrooms). Додатковими факторами, які можуть покращити прогноз у реальних умовах, є точна географічна локація, інфраструктура поруч та поточний стан побудови.

